In [1]:
import os
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings("ignore")

import torch
from transformers import DistilBertTokenizer, DistilBertModel

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay

import matplotlib.pyplot as plt

print("✅ Libraries loaded")

✅ Libraries loaded


In [2]:
# ── Mental Health dataset ──────────────────────────────────────────────────
df_mh = pd.read_csv("/kaggle/input/datasets/suchintikasarkar/sentiment-analysis-for-mental-health/Combined Data.csv")
df_mh.columns = ["id", "statement", "status"]
df_mh["status"] = df_mh["status"].str.strip()
TARGET_MH = ["Normal", "Depression", "Anxiety"]
df_mh = df_mh[df_mh["status"].isin(TARGET_MH)].dropna(subset=["statement"])
df_mh = df_mh[df_mh["statement"].str.strip() != ""]
df_mh = df_mh[["statement", "status"]].reset_index(drop=True)
print("✅ Mental Health dataset:")
print(df_mh["status"].value_counts())

# ── Stress dataset (Dreaddit) ──────────────────────────────────────────────
print("\nFiles in stress dataset:")
for f in os.listdir("/kaggle/input/datasets/mexwell/stress-detection-from-social-media-articles"):
    print(" ", f)

✅ Mental Health dataset:
status
Normal        16343
Depression    15404
Anxiety        3841
Name: count, dtype: int64

Files in stress dataset:
  Reddit_Combi.csv
  Reddit_Title.csv
  Twitter_Full.csv
  Twitter_ Non-Advert-Tabelle 1.csv


In [3]:
stress_path = "/kaggle/input/datasets/mexwell/stress-detection-from-social-media-articles/"
stress_files = os.listdir(stress_path)
print("Files:", stress_files)

dfs = []
for f in stress_files:
    if f.endswith(".csv"):
        tmp = pd.read_csv(stress_path + f, sep=';', on_bad_lines='skip')
        print(f"\n{f}: {tmp.shape}")
        print("Columns:", tmp.columns.tolist())
        print(tmp.head(2))
        dfs.append(tmp)

Files: ['Reddit_Combi.csv', 'Reddit_Title.csv', 'Twitter_Full.csv', 'Twitter_ Non-Advert-Tabelle 1.csv']

Reddit_Combi.csv: (3123, 5)
Columns: ['title', 'body', 'Body_Title', 'label', 'Unnamed: 4']
                                               title  \
0                     Envy to other is swallowing me   
1  Nothin outta the ordinary. Paradise. Job stres...   

                                                body  \
0  Im from developingcountry, Indonesia , and for...   
1  Um hello ....well many can relate im sure. Aft...   

                                          Body_Title  label  Unnamed: 4  
0  Envy to other is swallowing me Im from develop...      1         NaN  
1  Nothin outta the ordinary. Paradise. Job stres...      1         NaN  

Reddit_Title.csv: (5556, 5)
Columns: ['title', 'label', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']
                                               title  label  Unnamed: 2  \
0  My aunt and uncle scoring their first gig as p...      0         

In [4]:
# Filter stress
df_r_combi = pd.read_csv(stress_path + "Reddit_Combi.csv", sep=';', on_bad_lines='skip')
df_r_combi = df_r_combi[df_r_combi["label"] == 1][["Body_Title"]].copy()
df_r_combi.columns = ["statement"]

df_r_title = pd.read_csv(stress_path + "Reddit_Title.csv", sep=';', on_bad_lines='skip')
df_r_title = df_r_title[df_r_title["label"] == 1][["title"]].copy()
df_r_title.columns = ["statement"]

df_tw_full = pd.read_csv(stress_path + "Twitter_Full.csv", sep=';', on_bad_lines='skip')
df_tw_full = df_tw_full[df_tw_full["labels"] == 1][["text"]].copy()
df_tw_full.columns = ["statement"]

df_tw_non = pd.read_csv(stress_path + "Twitter_ Non-Advert-Tabelle 1.csv", sep=';', on_bad_lines='skip')
df_tw_non = df_tw_non[df_tw_non["label"] == 1][["text"]].copy()
df_tw_non.columns = ["statement"]

df_stress = pd.concat([df_r_combi, df_r_title, df_tw_full, df_tw_non], ignore_index=True)
df_stress["status"] = "Stress"
df_stress = df_stress.dropna(subset=["statement"])
df_stress = df_stress[df_stress["statement"].str.strip() != ""].reset_index(drop=True)

# Merge
df = pd.concat([df_mh, df_stress], ignore_index=True)

# Upsample
from sklearn.utils import resample
df_normal = df[df["status"] == "Normal"]
df_depression = df[df["status"] == "Depression"]
df_stress_up = df[df["status"] == "Stress"]
df_anxiety = df[df["status"] == "Anxiety"]

df_stress_up = resample(df_stress_up, replace=True, n_samples=15000, random_state=42)
df_anxiety_up = resample(df_anxiety, replace=True, n_samples=15000, random_state=42)

df = pd.concat([df_normal, df_depression, df_stress_up, df_anxiety_up], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print("✅ Balanced dataset:")
print(df["status"].value_counts())

✅ Balanced dataset:
status
Normal        16343
Depression    15404
Anxiety       15000
Stress        15000
Name: count, dtype: int64


In [5]:
def compute_dass21_scores(text):
    DEPRESSION_KEYWORDS = [
        "hopeless", "worthless", "empty", "no meaning", "nothing to live",
        "pointless", "can't go on", "want to die", "no future", "burden",
        "sad", "depressed", "miserable", "numb", "lifeless", "give up",
        "no energy", "no motivation", "crying", "tears", "lost", "alone"
    ]
    ANXIETY_KEYWORDS = [
        "anxious", "panic", "worried", "fear", "terrified", "nervous",
        "heart racing", "can't breathe", "shaking", "trembling", "dread",
        "scared", "phobia", "tense", "overwhelmed", "chest pain", "dizzy",
        "catastrophe", "danger", "threat", "uneasy", "restless"
    ]
    STRESS_KEYWORDS = [
        "stressed", "stress", "pressure", "overloaded", "burned out", "burnout",
        "deadline", "exhausted", "workload", "too much", "can't cope",
        "overwhelmed", "tension", "irritable", "frustrated", "no time",
        "rushing", "hectic", "swamped", "drained", "on edge"
    ]
    text_lower = text.lower()
    dep_score = sum(1 for kw in DEPRESSION_KEYWORDS if kw in text_lower)
    anx_score = sum(1 for kw in ANXIETY_KEYWORDS if kw in text_lower)
    str_score = sum(1 for kw in STRESS_KEYWORDS if kw in text_lower)
    return (
        min(dep_score / 10.0, 1.0),
        min(anx_score / 10.0, 1.0),
        min(str_score / 10.0, 1.0)
    )

scores = df["statement"].apply(compute_dass21_scores)
df["dass_depression"] = scores.apply(lambda x: x[0])
df["dass_anxiety"]    = scores.apply(lambda x: x[1])
df["dass_stress"]     = scores.apply(lambda x: x[2])

print("✅ DASS-21 scores computed")
print(df[["statement", "dass_depression", "dass_anxiety", "dass_stress"]].head())

✅ DASS-21 scores computed
                                           statement  dass_depression  \
0  Since I am such a piece of shit and just take ...              0.2   
1  Afraid i've been exposed to hantavirus, really...              0.0   
2  Just hoping i can drink enough to go to sleep ...              0.1   
3  last week i went on a spring break trip it wa ...              0.0   
4                                 Leaving abusive ex              0.0   

   dass_anxiety  dass_stress  
0           0.1          0.0  
1           0.4          0.0  
2           0.0          0.0  
3           0.2          0.0  
4           0.0          0.0  


In [6]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
bert_model = DistilBertModel.from_pretrained("distilbert-base-uncased")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_model = bert_model.to(device)
bert_model.eval()
print(f"✅ DistilBERT loaded | Device: {device}")

def get_embeddings(texts, batch_size=32):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(batch, padding=True, truncation=True,
                            max_length=128, return_tensors="pt").to(device)
        with torch.no_grad():
            output = bert_model(**encoded)
        cls = output.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls)
        if i % 3200 == 0:
            print(f"   {min(i+batch_size, len(texts))}/{len(texts)} done...")
    return np.vstack(all_embeddings)

bert_embeddings = get_embeddings(df["statement"].tolist())
print(f"✅ Embeddings shape: {bert_embeddings.shape}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ DistilBERT loaded | Device: cuda
   32/61747 done...
   3232/61747 done...
   6432/61747 done...
   9632/61747 done...
   12832/61747 done...
   16032/61747 done...
   19232/61747 done...
   22432/61747 done...
   25632/61747 done...
   28832/61747 done...
   32032/61747 done...
   35232/61747 done...
   38432/61747 done...
   41632/61747 done...
   44832/61747 done...
   48032/61747 done...
   51232/61747 done...
   54432/61747 done...
   57632/61747 done...
   60832/61747 done...
✅ Embeddings shape: (61747, 768)


In [7]:
dass_features = df[["dass_depression", "dass_anxiety", "dass_stress"]].values
X = np.hstack([bert_embeddings, dass_features])
print(f"✅ Hybrid features shape: {X.shape}")

le = LabelEncoder()
y = le.fit_transform(df["status"])
print(f"Classes: {le.classes_}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

✅ Hybrid features shape: (61747, 771)
Classes: ['Anxiety' 'Depression' 'Normal' 'Stress']
Train: (49397, 771) | Test: (12350, 771)


In [8]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
svm = LinearSVC(C=1.0, max_iter=2000, random_state=42, class_weight='balanced')
print("Running 5-Fold CV...")
cv_scores = cross_val_score(svm, X_train, y_train, cv=skf, scoring="accuracy", n_jobs=-1)
for i, s in enumerate(cv_scores, 1):
    print(f"  Fold {i}: {s:.4f}")
print(f"  Mean CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

svm_cal = CalibratedClassifierCV(
    LinearSVC(C=1.0, max_iter=2000, random_state=42, class_weight='balanced')
)
svm_cal.fit(X_train, y_train)
y_pred = svm_cal.predict(X_test)

print(f"\n✅ Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

Running 5-Fold CV...
  Fold 1: 0.8765
  Fold 2: 0.8671
  Fold 3: 0.8711
  Fold 4: 0.8671
  Fold 5: 0.8738
  Mean CV Accuracy: 0.8711 ± 0.0037

✅ Test Accuracy: 0.8723

Classification Report:
              precision    recall  f1-score   support

     Anxiety       0.89      0.91      0.90      3000
  Depression       0.87      0.86      0.87      3081
      Normal       0.87      0.92      0.90      3269
      Stress       0.85      0.79      0.82      3000

    accuracy                           0.87     12350
   macro avg       0.87      0.87      0.87     12350
weighted avg       0.87      0.87      0.87     12350

